# dscdemand: Getting Started

This notebook fits the storable-complementary demand model to the synthetic
example panel and computes the purchase response to an upstream shock.

The model: each week a household decides whether to buy a storable
**downstream** good (for example, hand lotion). Recent purchases of a
complementary **upstream** good (for example, hand sanitizer) build an
exposure stock $K$ that raises the value of buying; downstream inventory
$\xi$ lowers it. Households are forward-looking, so the probability of buying
comes from a dynamic program:

$$
u_1 = a + f(K) + \lambda\,\xi + \alpha\,\text{price} + b\,\text{event}, \qquad u_0 = 0,
$$

with $f$ piecewise linear in exposure and stocks that decay each week,
$K' = \eta K + S$ and $\xi' = \rho\,\xi + Q$.

## 1. Load a household-week panel

In [ ]:
from dscdemand import StorableComplementaryModel, load_example, multipliers

dataset = load_example()
dataset.data.head()

`load_example()` returns a `PanelDataset`. To use your own data, pass a
`pandas.DataFrame` and name its columns:

```python
from dscdemand import PanelDataset

dataset = PanelDataset(df, household="hh_id", week="week",
                       upstream="sanitizer_qty", purchase="bought_lotion",
                       volume="lotion_oz", price="lotion_price",
                       covariates=("event",))
```

## 2. Fit the model

The stock retention rates $\eta$ and $\rho$ and the discount factor $\beta$
are fixed; the utility parameters are estimated by nested fixed point
maximum likelihood. `event` is an unanticipated shock, so households do not
expect it when planning ahead.

In [ ]:
model = StorableComplementaryModel(eta=0.6, rho=0.8, unanticipated=("event",))
model.fit(dataset)
model.params_.round(3)

The example data were generated with $a = 1$, $\gamma_1 = 0.8$,
$\gamma_2 = 0.2$, $\lambda = -2.5$, $\alpha = -0.4$, and $b = 1$. Building the
grid from the data shifts the intercept and attenuates $\lambda$ somewhat
(see the README's limitations).

## 3. Response to an upstream shock

One extra upstream purchase (by default, the 75th percentile of positive
purchases) raises the probability of buying for a few weeks. Households that
buy early then hold more inventory and buy less later, so part of the extra
demand is offset.

In [ ]:
response = model.shock_response(weeks=26)
response.head(10).round(4)

In [ ]:
multipliers(response["response"])

- **gross**: extra purchases per household (positive weeks only);
- **net**: after later reductions;
- **offset_ratio**: share of the extra purchases that later weeks cancel.

## 4. More

- Unobserved household types: `StorableComplementaryModel(..., n_types=2)`
  fits a finite mixture by EM.
- Simulated data with known parameters: `model.simulate(params=[...])`.
- Storable-only and complementary-only models are planned
  (`dscdemand/models/`).